AI as a Service
---------------

Beispiel für einen **Ollama-Client**, der mit einem **Ollama Inference Server auf einem separaten Rechner** kommuniziert.

Auf diesem Server läuft **Ollama als System-Service** und verwaltet **zwei KI-Modelle**, die bei Bedarf geladen und wieder entladen werden. Die Modelle werden zentral betrieben und stehen allen Clients über eine **OpenAI-kompatible API** zur Verfügung.

Der folgende Code zeigt, wie aus einem **Python-Jupyter-Notebook** über diese OpenAI-kompatible API auf den betriebenen Ollama-Service zugegriffen wird.

Die Verbindung erfolgt über den API-Endpoint des Servers. Der verwendete API-Key dient dabei lediglich als Platzhalter.

Die Funktion `ask` kapselt einen einfachen Chat-Request und ermöglicht es, unterschiedliche KI-Modelle gezielt anzusprechen, ohne den übrigen Code anpassen zu müssen.


In [ ]:
from openai import OpenAI
import time

def ask(model, prompt, port=11434, max_tokens=2048):

    client = OpenAI(
        base_url=f"http://10.3.24.11:{port}/v1",
        api_key="sglang"
    )

    start = time.perf_counter()

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=max_tokens,
    )

    end = time.perf_counter()
    duration = end - start

    answer = response.choices[0].message.content

    prompt_tokens = response.usage.prompt_tokens
    completion_tokens = response.usage.completion_tokens
    total_tokens = response.usage.total_tokens

    tokens_per_sec = completion_tokens / duration

    print(f"Antwortzeit: {duration:.2f} Sekunden")
    print(f"Prompt Tokens: {prompt_tokens}")
    print(f"Completion Tokens: {completion_tokens}")
    print(f"Total Tokens: {total_tokens}")
    print(f"Decode Speed: {tokens_per_sec:.2f} tokens/s")

    return answer

Dieses Codebeispiel sendet eine Anfrage an das Allround-Sprachmodell `llama3.1:8b-instruct-q4_K_M`. Das Modell eignet sich für erklärende sowie allgemein technische Fragestellungen und demonstriert eine einfache Textabfrage über die OpenAI-kompatible Schnittstelle.

Bei der Interpretation der gemessenen Antwortzeit ist zu beachten, dass LLMs auf einem Ollama-Server je nach Betriebszustand unterschiedlich schnell reagieren. Insbesondere der erste Request nach dem Laden eines Modells fällt oft langsamer aus, weil das Modell zunächst in den Arbeitsspeicher beziehungsweise Grafikspeicher geladen und initialisiert werden muss. Dieser Effekt wird in der Praxis häufig als Kaltstart bezeichnet. Nachfolgende Anfragen an ein bereits geladenes Modell sind in der Regel schneller, da das Modell sich bereits in einem betriebsbereiten Zustand befindet. Die gemessene Antwortzeit hängt daher nicht nur von der eigentlichen Inferenz, sondern auch davon ab, ob bereits ein sogenannter Warm-Start vorliegt.


In [ ]:
print(ask(
    "llama3.1:8b-instruct-q4_K_M",
    "Erkläre HTTP Status Codes kurz."
))

Dieses Codebeispiel verwendet das Modell gemma3:12b, das sich besonders fuer strukturierte Antworten und Code-nahe Aufgaben eignet.

In [ ]:
print(ask(
    "gemma3:12b",
    "Schreibe ein kurzes Python Beispiel für einen REST Client."
))

Beide Codebeispiele zeigen, wie gezielt unterschiedliche Modelle fuer unterschiedliche Aufgaben angesprochen werden koennen.

- - -

Codebeispiel ergänzt um [Snippets für Masterprompt](https://gitlab.com/ch-tbz-it/Stud/allgemein/ki-kompetenz).

In [ ]:
ZUSATZPROMPT = """[Rolle]
Sei mein persönlicher Tutor und Lernbegleiter. Unterstütze mich aktiv beim selbstständigen Lernen, motiviere mich und fördere reflektiertes Denken.

[Ziel]
Hilf mir, die relevanten Grundlagen und Konzepte zu verstehen, ohne vollständige Lösungen für meine Aufgaben zu liefern. Führe mich durch den Lernprozess, indem du Denkwege, Vorgehensmethoden und Strategien aufzeigst, mit denen ich Probleme eigenständig bearbeiten kann.

Halte dich dabei an den Selbstlernzyklus, der im folgenden GitLab-Repository beschrieben ist: https://gitlab.com/ch-tbz-it/Stud/allgemein/SOL

[OUTPUT-FORMAT]
Strukturiere jede Antwort in exakt den folgenden vier Kategorien:

1) Grundlagen: Erkläre kurz, präzise und verständlich die für die Frage relevanten Basics (Begriffe, Regeln, typische Stolpersteine).  
2) Konzeptverständnis: Verdeutliche die zugrunde liegende Logik oder das Prinzip mit nachvollziehbarer Erklärung, kleinen Beispielen oder Analogien.  
3) Nächster Schritt: Beschreibe einen konkreten, durchführbaren nächsten Schritt (mit Mini-Checkliste oder Leitfragen), der mich der Lösung näherbringt, ohne sie direkt vorwegzunehmen.  
4) Nutzung Lernportfolio: Schlage einen konkreten Eintrag für das Lernportfolio vor (Ziel, Vorgehen, Erkenntnisse, offene Fragen, nächste Massnahmen). Wegleitung: https://gitlab.com/ch-tbz-it/Stud/allgemein/lernportfolio

Falls Ressourcen angehängt sind, nutze primär diese, um den thematischen Kontext zu erfassen. Wenn du zusätzliche externe Quellen verwendest, nenne sie transparent mit Titel und Link.

Bitte halte dich strikt an diese Struktur und das angegebene Niveau. Formuliere exakt, prüfe deine Antworten sorgfältig und denke vertieft nach, bevor du Schlussfolgerungen ziehst.
"""

AUFGABE = "Schreibe ein kurzes Python Beispiel für einen REST Client."

print(ask(
    "gemma3:12b",
    f"{ZUSATZPROMPT}\n\n[Aufgabe]\n{AUFGABE}"
))

- - - 

Ergänzt in eurer **WireGuard-Konfiguration** die Einstellung `PersistentKeepalive = 25`. Installiert anschliessend **Ollama** und aktiviert in den Einstellungen die Option **„Expose Ollama to the network“**.

Passt danach in der Funktion `ask` die IP-Adresse an eure eigene Adresse an und führt die Beispiele erneut aus.


- - - 

In der Funktion `ask` wird der eigentliche Chat-Request an den Ollama-Server aufgebaut:

```python
response = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0.2,
)
```

Der Eintrag `model=model` legt fest, welches Modell für die Anfrage verwendet wird. Dadurch kann dieselbe Funktion mit unterschiedlichen Modellen genutzt werden, ohne dass der übrige Code angepasst werden muss.

Über `messages=[{"role": "user", "content": prompt}]` wird die eigentliche Benutzereingabe an das Modell übergeben. Das Format entspricht der OpenAI-kompatiblen Chat-Schnittstelle: Jede Nachricht besitzt eine Rolle, hier `user`, sowie den eigentlichen Inhalt in `content`.

Der Parameter `temperature=0.2` steuert, wie stark das Modell bei der Antwort variiert. Niedrige Werte führen in der Regel zu stabileren, präziseren und besser reproduzierbaren Antworten. Höhere Werte erzeugen oft kreativere, aber auch weniger konsistente Resultate.

Ergänzt den Request nun zusätzlich um eine Begrenzung der Antwortlänge über `max_tokens` und variiert anschliessend sowohl `temperature` als auch `max_tokens`. Führt die Beispiele danach erneut aus und vergleicht, wie sich diese Parameter auf Antwortstil, Antwortlänge und Antwortzeit auswirken.

Eine mögliche Erweiterung sieht zum Beispiel so aus:

```python
response = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0,
    max_tokens=300,
)
```

Für erste Vergleiche bietet es sich an, mit mehreren Kombinationen zu experimentieren, etwa mit einer tiefen Temperatur für sachliche Antworten und einer höheren Temperatur für offenere oder kreativere Formulierungen. Ebenso lässt sich mit kleineren und grösseren `max_tokens` beobachten, wie stark die Länge der Modellantwort begrenzt wird.

Darüber hinaus gibt es weitere Möglichkeiten zur Optimierung. Relevant sind insbesondere die Wahl eines passenden Modells, die Quantisierung, das Warmhalten häufig genutzter Modelle, die Grösse und Struktur des Prompts sowie die verfügbare Hardware auf dem Server. Auch die Netzwerklatenz, parallele Zugriffe mehrerer Clients und das Ladeverhalten der Modelle können die gemessene Antwortzeit deutlich beeinflussen.
